# DPGP3 TE matrix preview (Lee et al., 2021)

Manuscript Figs. 5–6: mean–variance–ρ scatter and per-family copy-number histograms.

Loads `DPGP3_TE_insertion_depletion_summary_everything_masked_noHet_bool.txt` (same folder) and writes figures alongside this notebook.

**Run from** `figure5/` or `TE_overdispersion/` (paths are resolved in the first code cell).

**NA handling:** set `NA_POLICY` to `"missing"` (recommended), `"as_present"`, or `"as_absent"`.


In [ ]:
from pathlib import Path

import pandas as pd

# NA handling for ZI* strain columns.
# "missing"     — keep file NA as NaN (NA is unobserved; sums ignore it by default).
# "as_present"  — impute NA as 1 (present TE); sensitivity / upward bias where masked.
# "as_absent"   — impute NA as 0 (absent); sensitivity / downward bias where masked.
NA_POLICY = "missing"
assert NA_POLICY in ("missing", "as_present", "as_absent")

# Paths: data and figures in figure5/; csv not used here
DATA_NAME = "DPGP3_TE_insertion_depletion_summary_everything_masked_noHet_bool.txt"
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if not (SCRIPT_DIR / DATA_NAME).is_file():
    for candidate in (
        Path.cwd() / "figure5",
        Path.cwd() / "TE_overdispersion" / "figure5",
    ):
        if (candidate / DATA_NAME).is_file():
            SCRIPT_DIR = candidate
            break
DATA_PATH = SCRIPT_DIR / DATA_NAME
OUTPUT_DIR = SCRIPT_DIR

# The file starts with a comment line beginning with '#', then has a tab-separated header.
df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    comment="#",
    na_values=["NA"],
    dtype={"TE": str, "family": str},
)

zi_cols = [c for c in df.columns if str(c).startswith("ZI")]
if NA_POLICY == "as_present":
    df[zi_cols] = df[zi_cols].fillna(1)
elif NA_POLICY == "as_absent":
    df[zi_cols] = df[zi_cols].fillna(0)

print(f"NA_POLICY={NA_POLICY!r}  ({len(zi_cols)} ZI columns)")
print(f"DATA_PATH={DATA_PATH}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")


In [ ]:
print("Loaded shape:", df.shape)
# Show the first few rows (including the first TE interval and family label).
df.head(10)


In [ ]:
# Remove ambiguous 'multi' family rows (all strains; before call-rate filtering).
df_nonmulti = df.loc[df["family"] != "multi"].copy()

print("Non-multi shape (all strains):", df_nonmulti.shape)
df_nonmulti.head(10)


In [ ]:
# Per-strain missingness on non-multi sites (next cell drops call_rate < CALL_RATE_MIN).
# If NA_POLICY is as_present/as_absent, ZI columns have no NaNs here → n_missing=0, call_rate=1 for all.
from IPython.display import display

zi_cols = [c for c in df_nonmulti.columns if c.startswith("ZI")]
M = df_nonmulti[zi_cols].isna().sum()
C = df_nonmulti[zi_cols].notna().sum()
n_sites = len(df_nonmulti)
strain_miss = (
    pd.DataFrame({"n_missing": M, "n_calls": C})
    .assign(call_rate=lambda x: x["n_calls"] / (x["n_calls"] + x["n_missing"]))
    .sort_values("n_missing", ascending=False)
)
print(f"Non-multi TE sites (rows): {n_sites}")
print("call_rate = n_calls / (n_calls + n_missing) across those sites")
print("\ncall_rate across strains:")
print(strain_miss["call_rate"].describe())
print("\nTop 15 strains by missing count:")
display(strain_miss.head(15))
# Lee et al. removed strains with >4000 missing TE calls (different pipeline; for scale only)
lee_threshold = 4000
n_over = (strain_miss["n_missing"] > lee_threshold).sum()
print(f"\nStrains with n_missing > {lee_threshold} (Lee-style outlier scale): {n_over}")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(strain_miss["call_rate"], bins=30, edgecolor="k", alpha=0.7)
ax.set_xlabel("call rate")
ax.set_ylabel("# strains")
ax.set_title("Per-strain call rate (non-multi sites)")
plt.tight_layout()
plt.show()


In [ ]:
# Call-rate filter (may drop nothing if all strains pass CALL_RATE_MIN)
CALL_RATE_MIN = 0.8
passing = set(strain_miss[strain_miss["call_rate"] >= CALL_RATE_MIN].index)
all_zi = [c for c in df_nonmulti.columns if c.startswith("ZI")]
keep_strains = [c for c in all_zi if c in passing]
dropped_callrate = [c for c in all_zi if c not in passing]
meta_cols = ["TE", "family"]
clean_df = df_nonmulti[meta_cols + keep_strains].copy()
zi_cols = [c for c in clean_df.columns if c.startswith("ZI")]
print(f"CALL_RATE_MIN = {CALL_RATE_MIN}")
print(f"N dropped (call_rate < {CALL_RATE_MIN}): {len(dropped_callrate)}")
print("Dropped strains:", dropped_callrate)
print(f"Strains present after call-rate filter: {len(zi_cols)}")
print("Shape (clean_df):", clean_df.shape)


In [ ]:
# TE family classification: coarse order (LTR / non-LTR / DNA)
# Edit this mapping if you want to refine/extend classifications.
# Families not listed here get te_order "Unknown".

te_order_map = {
    # LTR retrotransposons
    "412": "LTR",
    "BLOOD": "LTR",
    "COPIA": "LTR",
    "GYPSY": "LTR",
    "GYPSY1": "LTR",
    "GYPSY2": "LTR",
    "GYPSY3": "LTR",
    "GYPSY4": "LTR",
    "GYPSY5": "LTR",
    "GYPSY6": "LTR",
    "GYPSY7": "LTR",
    "GYPSY8": "LTR",
    "GYPSY9": "LTR",
    "GYPSY10": "LTR",
    "GYPSY11": "LTR",
    "GYPSY12": "LTR",
    "GYPSY13": "LTR",
    "GYPSY14": "LTR",
    "GYPSY15": "LTR",
    "IDEFIX": "LTR",
    "INVADER": "LTR",
    "INVADER2": "LTR",
    "INVADER3": "LTR",
    "INVADER4": "LTR",
    "MDG1": "LTR",
    "MDG3": "LTR",
    "MICROPIA": "LTR",
    "ROO": "LTR",
    "STALKER": "LTR",
    "TABOR": "LTR",
    "ZAM": "LTR",
    "TRANSPAC": "LTR",
    # 297: LTR / gypsy-group retrotransposon (D. melanogaster; FlyBase natural transposon 297).
    "297": "LTR",
    # TIRANT: gypsy-family LTR retrotransposon (D. melanogaster).
    "TIRANT": "LTR",
    # OPUS: LTR retrotransposon family (expressed with copia/gypsy-class elements in D. melanogaster).
    "OPUS": "LTR",

    # non-LTR retrotransposons (LINE-like)
    "DOC": "non-LTR",
    "DOC2": "non-LTR",
    "DOC3": "non-LTR",
    "DOC4": "non-LTR",
    "F": "non-LTR",
    "I": "non-LTR",
    "JOCKEY": "non-LTR",
    "JOCKEY2": "non-LTR",
    "R1": "non-LTR",
    "R2": "non-LTR",
    # BS: LINE-like non-LTR retrotransposon (D. melanogaster; FlyBase BS-element).
    "BS": "non-LTR",
    # JUAN (JuanDm): non-LTR retrotransposon (D. melanogaster).
    "JUAN": "non-LTR",

    # DNA transposons
    # 1360: DNA transposon (D. melanogaster; FlyBase natural transposon 1360).
    "1360": "DNA",
    "BARI": "DNA",
    "BARI1": "DNA",
    "BARI2": "DNA",
    "BURDOCK": "DNA",
    "HOBO": "DNA",
    "HOPPER": "DNA",
    "HMS-BEAGLE": "DNA",
    "HMS-BEAGLE2": "DNA",
    "P": "DNA",
    "POGO": "DNA",
    "TC1-2": "DNA",
    "TRANSIB": "DNA",
    "TRANSIB2": "DNA",
    "TRANSIB4": "DNA",
    # FB: foldback / foldback-type DNA transposon (D. melanogaster).
    "FB": "DNA",
    # S (S-element): terminal-inverted-repeat (TIR) DNA transposon (D. melanogaster).
    "S": "DNA",
}

families_in_data = sorted(clean_df["family"].unique())
unknown_families = sorted([f for f in families_in_data if te_order_map.get(f) is None])

print("Mapped families:", len(te_order_map))
print("Families in data:", len(families_in_data))
print("Unmapped families (te_order=Unknown):", len(unknown_families))
print(unknown_families[:50], "..." if len(unknown_families) > 50 else "")


In [ ]:
# Per-family burden moments across individuals (ignoring NaN)
# Burden for a given (family, individual) = number of sites with value==1 for that family in that individual.

filtered_df = clean_df

# n_sites = number of TE sites/rows in the table for each family
family_copies = filtered_df.groupby("family").size()

# Total TE presences per family across all individuals (sum of 1s, ignoring NaN).
# This is the sum of burdens across individuals for each family.
family_total_present = filtered_df.groupby("family")[zi_cols].sum().sum(axis=1)

# Sum across sites within each family for each individual (NaNs are ignored by default).
family_by_individual = filtered_df.groupby("family")[zi_cols].sum()

stats = pd.DataFrame({
    "n_sites": family_copies,
    "n_present": family_total_present,
    "n_individuals": family_by_individual.notna().sum(axis=1),
    "mean": family_by_individual.mean(axis=1),
    "variance": family_by_individual.var(axis=1),
    "skewness": family_by_individual.apply(lambda s: s.dropna().skew(), axis=1),
    # pandas kurtosis is Fisher by default => excess kurtosis
    "excess_kurtosis": family_by_individual.apply(lambda s: s.dropna().kurt(), axis=1),
})

stats["rho"] = stats["variance"] / stats["mean"]

# If mean==0, rho is undefined; set to NA for readability.
stats.loc[stats["mean"] == 0, "rho"] = pd.NA

# Coarse TE order classification (LTR / non-LTR / DNA).
# `te_order_map` is defined in the standalone classification cell above.
stats["te_order"] = stats.index.map(lambda fam: te_order_map.get(fam, "Unknown"))

# Put the classification right next to the family label in outputs.
stats = stats[[
    "te_order",
    "n_sites",
    "n_present",
    "n_individuals",
    "mean",
    "variance",
    "rho",
    "skewness",
    "excess_kurtosis",
]]

stats.sort_values("rho", ascending=False).head(20)


In [ ]:
# Count over/under/equidispersion by family based on rho = variance/mean
# Apply information filters so dispersion labels are interpretable.

min_n_sites = 0
min_n_present = 500

stats_f = stats.loc[(stats["n_sites"] >= min_n_sites) & (stats["n_present"] >= min_n_present)].copy()
rho = stats_f["rho"].dropna().astype(float)

eps = 1e-2  # tolerance for treating rho as ~1
n_over = int((rho > 1 + eps).sum())
n_under = int((rho < 1 - eps).sum())
n_equi = int((rho.sub(1).abs() <= eps).sum())

print("Filter: n_sites >=", min_n_sites, "and n_present >=", min_n_present)
print("Families passing filter:", int(stats_f.shape[0]))
print("Families with defined rho (filtered):", int(rho.shape[0]))
print("Overdispersed TE families (rho > 1):", n_over)
print("Underdispersed TE families (rho < 1):", n_under)
print("~Equidispersed TE families (|rho-1| <= eps):", n_equi)
print("eps used:", eps)

cols_to_show = [
    "te_order",
    "n_sites",
    "n_present",
    "mean",
    "variance",
    "rho",
    "skewness",
    "excess_kurtosis",
]

over_df = stats_f.loc[rho[rho > 1 + eps].index, cols_to_show].sort_values("rho", ascending=False)
under_df = stats_f.loc[rho[rho < 1 - eps].index, cols_to_show].sort_values("rho", ascending=True)
equi_df = stats_f.loc[rho[rho.sub(1).abs() <= eps].index, cols_to_show].sort_values("rho", ascending=True)


print("\nOverdispersed families (filtered):")
display(over_df)

print("\nUnderdispersed families (filtered):")
display(under_df)

print("\nEquidispersed families (filtered):")
display(equi_df)


In [ ]:
# Plot copy-number distributions for representative TE families (POGO, JOCKEY, MDG1)
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from scipy.stats import gaussian_kde

families_to_plot = ["POGO", "JOCKEY", "MDG1"]

class_colors = {
    "DNA": "#2E7D32",      # green
    "non-LTR": "#1E5AA8",  # blue
    "LTR": "#EF6C00",      # orange
    "Unknown": "#6D6D6D",  # gray
}


def _stats_box_anchor(ax, counts, bin_edges):
    """Axes coords + ha/va: pick upper-left or upper-right with least histogram overlap."""
    n = len(counts)
    if n == 0:
        return (0.98, 0.98, "right", "top")
    total = float(counts.sum())
    if total <= 0:
        return (0.98, 0.98, "right", "top")
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    ymax = float(counts.max())
    if ymax <= 0:
        ymax = 1.0
    xmin, xmax = ax.get_xlim()
    w = xmax - xmin
    if w <= 0:
        return (0.98, 0.98, "right", "top")

    def strip_scores(x_lo, x_hi):
        m = (centers >= x_lo) & (centers <= x_hi)
        if not m.any():
            return 0.0, 0.0
        c = counts[m]
        return float(c.sum()) / total, float(c.max()) / ymax

    ml, pl = strip_scores(xmin, xmin + 0.42 * w)
    mr, pr = strip_scores(xmin + 0.58 * w, xmax)

    candidates = [
        (0.02, 0.98, "left", "top", ml * (0.4 + 0.6 * pl)),
        (0.98, 0.98, "right", "top", mr * (0.4 + 0.6 * pr)),
    ]
    best = min(candidates, key=lambda t: t[4])
    return best[0], best[1], best[2], best[3]


fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for i, (ax, fam) in enumerate(zip(axes, families_to_plot)):
    if fam not in family_by_individual.index:
        ax.set_title(fam, fontweight="bold")
        ax.text(
            0.5,
            0.5,
            "Family not found",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontweight="bold",
        )
        ax.axis("off")
        continue

    x = family_by_individual.loc[fam].dropna().astype(float)

    te_class = stats.loc[fam, "te_order"] if fam in stats.index else "Unknown"
    color = class_colors.get(te_class, class_colors["Unknown"])

    mu = float(stats.loc[fam, "mean"]) if fam in stats.index else float(x.mean())
    var = float(stats.loc[fam, "variance"]) if fam in stats.index else float(x.var())
    skew = float(stats.loc[fam, "skewness"]) if fam in stats.index else float(x.skew())
    ex_kurt = float(stats.loc[fam, "excess_kurtosis"]) if fam in stats.index else float(x.kurt())
    rho_val = stats.loc[fam, "rho"] if fam in stats.index else (var / mu if mu != 0 else pd.NA)
    rho_str = "NA" if rho_val is None or pd.isna(rho_val) else f"{float(rho_val):.2f}"

    counts, bin_edges, _ = ax.hist(
        x, bins=15, color=color, alpha=0.55, edgecolor="black", linewidth=2.0
    )
    if len(x) >= 2 and float(x.var()) > 0:
        bin_width = float(bin_edges[1] - bin_edges[0])
        kde = gaussian_kde(x)
        xs = np.linspace(float(x.min()), float(x.max()), 200)
        ax.plot(
            xs,
            kde(xs) * len(x) * bin_width,
            color=color,
            linewidth=2.0,
            zorder=3,
        )
    tx, ty, tha, tva = _stats_box_anchor(ax, counts, bin_edges)

    ax.set_title(fam, fontweight="bold")
    ax.set_xlabel("Copy Number", fontweight="bold")
    if i == 0:
        ax.set_ylabel("Frequency", fontweight="bold")
    else:
        ax.set_ylabel("")
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")

    box_txt = "\n".join([
        f"μ = {mu:.2f}",
        f"σ² = {var:.2f}",
        f"Skew = {skew:.2f}",
        f"Ex. Kurt. = {ex_kurt:.2f}",
        f"ρ = {rho_str}",
    ])
    ax.text(
        tx,
        ty,
        box_txt,
        transform=ax.transAxes,
        ha=tha,
        va=tva,
        fontsize=9,
        fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="black", alpha=0.9),
    )

plt.tight_layout()
# Figures saved under OUTPUT_DIR (defined in first cell).
_out_pdf = OUTPUT_DIR / "DPGP3_TE_copynumber_POGO_JOCKEY_MDG1.pdf"
# fig.savefig(_out_pdf, bbox_inches="tight")
print("Saved:", _out_pdf.resolve())
plt.show()


In [ ]:
# All TE families that passed the filter (same stats_f as above); includes P, I, ROO.
# 4×8 grid, histogram colours match te_order (LTR / non-LTR / DNA / Unknown).
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import gaussian_kde

plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 160

class_colors = {
    "DNA": "#2E7D32",
    "non-LTR": "#1E5AA8",
    "LTR": "#EF6C00",
    "Unknown": "#6D6D6D",
}


def _stats_box_anchor(ax, counts, bin_edges):
    """Axes coords + ha/va: pick upper-left or upper-right with least histogram overlap."""
    n = len(counts)
    if n == 0:
        return (0.98, 0.98, "right", "top")
    total = float(counts.sum())
    if total <= 0:
        return (0.98, 0.98, "right", "top")
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    ymax = float(counts.max())
    if ymax <= 0:
        ymax = 1.0
    xmin, xmax = ax.get_xlim()
    w = xmax - xmin
    if w <= 0:
        return (0.98, 0.98, "right", "top")

    def strip_scores(x_lo, x_hi):
        m = (centers >= x_lo) & (centers <= x_hi)
        if not m.any():
            return 0.0, 0.0
        c = counts[m]
        return float(c.sum()) / total, float(c.max()) / ymax

    ml, pl = strip_scores(xmin, xmin + 0.42 * w)
    mr, pr = strip_scores(xmin + 0.58 * w, xmax)

    candidates = [
        (0.02, 0.98, "left", "top", ml * (0.4 + 0.6 * pl)),
        (0.98, 0.98, "right", "top", mr * (0.4 + 0.6 * pr)),
    ]
    best = min(candidates, key=lambda t: t[4])
    return best[0], best[1], best[2], best[3]


all_fams = sorted(stats_f.index.tolist())
nrows, ncols = 8, 4
per_fig = nrows * ncols

print(f"stats_f families: {len(all_fams)} | panels per figure: {per_fig}")

# Figures saved under OUTPUT_DIR (defined in first cell).
_pdf_out = OUTPUT_DIR / "DPGP3_TE_all_families_hist.pdf"
pdf = PdfPages(_pdf_out)

for start in range(0, len(all_fams), per_fig):
    chunk = all_fams[start : start + per_fig]
    # Only use the rows we need on the last page (avoids a big blank band under the final row).
    nrows_page = max(1, (len(chunk) + ncols - 1) // ncols)
    fig, axes = plt.subplots(
        nrows_page,
        ncols,
        figsize=(ncols * 2.2, nrows_page * 2.0),
        dpi=160,
        sharey=False,
        squeeze=False,
    )
    axes = axes.ravel()
    last_plot_row = (len(chunk) - 1) // ncols
    n_last = len(chunk) % ncols
    if n_last == 0:
        n_last = ncols
    for j, ax in enumerate(axes):
        if j >= len(chunk):
            ax.axis("off")
            continue
        fam = chunk[j]
        if fam not in family_by_individual.index:
            ax.set_title(fam, fontsize=8, fontweight="bold")
            ax.text(
                0.5,
                0.5,
                "Family not found",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=7,
                fontweight="bold",
            )
            ax.axis("off")
            continue

        x = family_by_individual.loc[fam].dropna().astype(float)
        te_class = stats.loc[fam, "te_order"] if fam in stats.index else "Unknown"
        color = class_colors.get(te_class, class_colors["Unknown"])

        mu = float(stats.loc[fam, "mean"]) if fam in stats.index else float(x.mean())
        var = float(stats.loc[fam, "variance"]) if fam in stats.index else float(x.var())
        skew = float(stats.loc[fam, "skewness"]) if fam in stats.index else float(x.skew())
        ex_kurt = float(stats.loc[fam, "excess_kurtosis"]) if fam in stats.index else float(x.kurt())
        rho_val = stats.loc[fam, "rho"] if fam in stats.index else (var / mu if mu != 0 else pd.NA)
        rho_str = "NA" if rho_val is None or pd.isna(rho_val) else f"{float(rho_val):.2f}"

        counts, bin_edges, _ = ax.hist(
            x, bins=15, color=color, alpha=0.55, edgecolor="black", linewidth=1.2
        )
        if len(x) >= 2 and float(x.var()) > 0:
            bin_width = float(bin_edges[1] - bin_edges[0])
            kde = gaussian_kde(x)
            xs = np.linspace(float(x.min()), float(x.max()), 200)
            ax.plot(
                xs,
                kde(xs) * len(x) * bin_width,
                color=color,
                linewidth=1.2,
                zorder=3,
            )
        tx, ty, tha, tva = _stats_box_anchor(ax, counts, bin_edges)

        col = j % ncols
        row = j // ncols

        ax.set_title(fam, fontsize=8, fontweight="bold")
        if row == last_plot_row:
            ax.set_xlabel("Copy Number", fontsize=6, fontweight="bold")
        elif n_last < ncols and row == last_plot_row - 1 and col >= n_last:
            # Last row is short: panels above empty bottom-right slots need the label too (e.g. ROO).
            ax.set_xlabel("Copy Number", fontsize=6, fontweight="bold")
        else:
            ax.set_xlabel("")
        if col == 0:
            ax.set_ylabel("Frequency", fontsize=6, fontweight="bold")
        else:
            ax.set_ylabel("")
        ax.tick_params(axis="both", labelsize=5)
        for label in ax.get_xticklabels():
            label.set_fontweight("bold")
        for label in ax.get_yticklabels():
            label.set_fontweight("bold")

        box_txt = "\n".join(
            [
                f"μ = {mu:.2f}",
                f"σ² = {var:.2f}",
                f"Skew = {skew:.2f}",
                f"Ex. Kurt. = {ex_kurt:.2f}",
                f"ρ = {rho_str}",
            ]
        )
        ax.text(
            tx,
            ty,
            box_txt,
            transform=ax.transAxes,
            ha=tha,
            va=tva,
            fontsize=4.5,
            fontweight="bold",
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="black", linewidth=0.4, alpha=0.92),
        )

    # Tight margins; trim bottom pad so the last row sits near the page edge.
    fig.tight_layout(pad=0.15, h_pad=0.2, w_pad=0.2)
    fig.subplots_adjust(bottom=0.02)
    # pdf.savefig(fig, bbox_inches="tight", pad_inches=0)
    plt.show()
    plt.close(fig)

pdf.close()
# print("Saved PDF:", _pdf_out.resolve())


In [ ]:
# Plot filtered families only (stats_f), with Unknown shown in grey
# Left panel: shaded from Poisson line up to slightly above highest observed variance
# Right panel: linear y-axis, no high-overdispersion red line

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plot_stats = stats_f.copy()
plot_stats = plot_stats[
    (plot_stats["mean"] > 0)
    & plot_stats["variance"].notna()
    & plot_stats["rho"].notna()
].copy()

palette = {
    "DNA": "#2E8B57",      # green
    "LTR": "#FF6B35",      # orange
    "non-LTR": "#4472C4",  # blue
    "Unknown": "#9E9E9E",  # grey
}

plot_stats["te_order"] = plot_stats["te_order"].fillna("Unknown")
plot_stats["color"] = plot_stats["te_order"].map(palette).fillna(palette["Unknown"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=130)

# Left panel: variance vs mean (log-log).
x = plot_stats["mean"].astype(float)
y = plot_stats["variance"].astype(float)

ax1.scatter(x, y, s=45, c=plot_stats["color"], edgecolor="k", linewidth=1.5, alpha=0.9)

xline = np.logspace(np.log10(max(x.min(), 1e-3)), np.log10(x.max() * 1.3), 300)
ax1.plot(xline, xline, "k--", lw=1.5)  # Poisson line

# Slanted band parallel to the Poisson line.
# anchor at the highest-variance point
i_top = y.idxmax()
x_top = float(x.loc[i_top])
y_top = float(y.loc[i_top])
# multiplier for upper boundary: y = k*x
# use 1.03 for "tiny bit on top"; use 1.15 if you want exact touch
k = max(1.0, 1.15 * (y_top / x_top))
# shade between y=x and y=kx
ax1.fill_between(
  xline,
  xline,
  k * xline,
  color="#f3b6b6",
  alpha=0.27,
)

ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.minorticks_off()
ax1.set_xlabel("Mean Copy Number (log10)", fontweight="bold")
ax1.set_ylabel("Variance in Copy Number (log10)", fontweight="bold")
ax1.grid(alpha=0.2, linestyle="--", which="major")

for _lbl in ax1.get_xticklabels():
    _lbl.set_fontweight("bold")
for _lbl in ax1.get_yticklabels():
    _lbl.set_fontweight("bold")

_has_unknown = (plot_stats["te_order"] == "Unknown").any()
_handles1 = [
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor=palette["DNA"], markeredgecolor="k", label="DNA transposons"),
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor=palette["LTR"], markeredgecolor="k", label="LTR transposons"),
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor=palette["non-LTR"], markeredgecolor="k", label="Non-LTR transposons"),
]
if _has_unknown:
    _handles1.append(
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=palette["Unknown"], markeredgecolor="k", label="Unknown"),
    )
_handles1.extend(
    [
        Line2D([0], [0], color="k", linestyle="--", lw=1.5, label="Poisson (Variance = Mean)"),
        Patch(facecolor="#f3b6b6", edgecolor="none", alpha=0.35, label="Moderate overdispersion"),
    ]
)
ax1.legend(
    handles=_handles1,
    loc="upper left",
    framealpha=0.9,
    prop={"size": 8, "weight": "bold"},
)

# Right panel: dispersion ratio rho = variance/mean.
bar_df = plot_stats.sort_values("rho", ascending=True)
ax2.bar(bar_df.index, bar_df["rho"], color=bar_df["color"], edgecolor="white", linewidth=0.5)
ax2.axhline(1.0, color="k", linestyle="--", lw=1.5)

ax2.set_ylabel(r"Variance/Mean ($\rho$)", fontweight="bold")
ax2.set_xlabel("Transposable elements", fontweight="bold")
ax2.set_ylim(0, float(bar_df["rho"].max()) * 1.08)
ax2.tick_params(axis="x", labelrotation=90, labelsize=6)
for _lbl in ax2.get_xticklabels():
    _lbl.set_fontweight("bold")
for _lbl in ax2.get_yticklabels():
    _lbl.set_fontweight("bold")

_handles2 = [
    Patch(facecolor=palette["DNA"], edgecolor="none", label="DNA transposons"),
    Patch(facecolor=palette["LTR"], edgecolor="none", label="LTR transposons"),
    Patch(facecolor=palette["non-LTR"], edgecolor="none", label="Non-LTR transposons"),
]
if _has_unknown:
    _handles2.append(Patch(facecolor=palette["Unknown"], edgecolor="none", label="Unknown"))
_handles2.append(Line2D([0], [0], color="k", linestyle="--", lw=1.5, label="Poisson expectation"))
ax2.legend(
    handles=_handles2,
    loc="upper left",
    framealpha=0.9,
    prop={"size": 8, "weight": "bold"},
)

plt.tight_layout()
# Figures saved under OUTPUT_DIR (defined in first cell).
_out_pdf = OUTPUT_DIR / "DPGP3_TE_mean_variance_rho.pdf"
# fig.savefig(_out_pdf, bbox_inches="tight")
print("Saved:", _out_pdf.resolve())
plt.show()
